# Data Cleaning -- High PDL1 1L Pembrolizumab vs. Chemotherapy
**This notebook prepares Flatiron Health CSV files for patients with advanced non-small cell lung cancer with high PDL1 (>= 50%) treated with first-line pembrolizumab or chemotherapy. Patients that are EGFR or ALK positive are excluded. Each CSV is cleaned using the flatiron_cleaner package. The cleaned dataframes are then merged into a single dataset, which will be used for survival analysis.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorNSCLC
from flatiron_cleaner import merge_dataframes

## Import data

In [2]:
df = pd.read_csv('../outputs/pembro_chemo_index.csv')

In [3]:
df.head(3)

,PatientID,LineName,StartDate
0,F998F69458EED,platinum,2024-01-10
1,F05B7A9AEA213,platinum,2019-10-30
2,F404B3C8249FE,platinum,2017-10-09


In [4]:
df.shape

(34491, 3)

In [5]:
ids = df.PatientID.to_list()

In [6]:
len(ids)

34491

## Clean CSV files 

In [7]:
# Initialize class 
processor = DataProcessorNSCLC()

### Process Enhanced_AdvancedNSCLC.csv

In [8]:
enhanced_df = processor.process_enhanced(file_path = '../../aNSCLC/data/Enhanced_AdvancedNSCLC.csv',
                                         patient_ids = ids,
                                         drop_dates = False)

2026-05-06 00:17:02,013 - INFO - Successfully read Enhanced_AdvancedNSCLC.csv file with shape: (114756, 6) and unique PatientIDs: 114756
2026-05-06 00:17:02,013 - INFO - Filtering for 34491 specific PatientIDs
2026-05-06 00:17:02,027 - INFO - Successfully filtered Enhanced_AdvancedNSCLC.csv file with shape: (34491, 6) and unique PatientIDs: 34491
2026-05-06 00:17:02,051 - INFO - Successfully processed Enhanced_AdvancedNSCLC.csv file with final shape: (34491, 8) and unique PatientIDs: 34491


In [9]:
df['StartDate'] = pd.to_datetime(df['StartDate'])

enhanced_df = pd.merge(enhanced_df, df[['PatientID', 'StartDate']], on = 'PatientID', how = 'left')
enhanced_df['days_adv_to_treatment'] = (enhanced_df['StartDate'] - enhanced_df['AdvancedDiagnosisDate']).dt.days
enhanced_df['days_adv_to_treatment'] = np.where(enhanced_df['days_adv_to_treatment'] < 0 , 0, enhanced_df['days_adv_to_treatment'])
enhanced_df['days_to_treatment_before_30d'] = np.where(enhanced_df['days_adv_to_treatment'] < 30, 1, 0)

In [10]:
enhanced_df['days_diagnosis_to_adv'] = np.where(enhanced_df['days_diagnosis_to_adv'] < 0 , 0, enhanced_df['days_diagnosis_to_adv'])
enhanced_df['days_diagnosis_to_adv'] = enhanced_df['days_diagnosis_to_adv'].fillna(0)

In [11]:
enhanced_df.GroupStage_mod.value_counts(dropna = False)

GroupStage_mod
IV         21396
III         8004
I           2862
II          1511
unknown      711
0              7
Name: count, dtype: int64

In [12]:
enhanced_df['GroupStage_mod'] = enhanced_df["GroupStage_mod"].map({
    '0': 0, 
    'I': 1,
    'II': 2,
    'III': 3,
    'IV': 4,
    'unknown': np.nan
})

enhanced_df['GroupStage_mod_na'] = np.where(enhanced_df['GroupStage_mod'].isna(), 1, 0)

# impute 4 since stage IV is most common
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].fillna(4)

In [13]:
enhanced_df['adv_diagnosis_year'] = pd.to_numeric(enhanced_df['adv_diagnosis_year'])
enhanced_df['before_2020'] = np.where(enhanced_df['adv_diagnosis_year'] < 2020, 1, 0)

In [14]:
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate', 
                                          'AdvancedDiagnosisDate', 
                                          'StartDate'])

### Process Demographics.csv 

In [15]:
demographics_df = processor.process_demographics(file_path = '../../aNSCLC/data/Demographics.csv',
                                                 index_date_df = df,
                                                 index_date_column = 'StartDate')

2026-05-06 00:17:02,164 - INFO - Successfully read Demographics.csv file with shape: (114756, 6) and unique PatientIDs: 114756
2026-05-06 00:17:02,216 - INFO - Successfully processed Demographics.csv file with final shape: (34491, 6) and unique PatientIDs: 34491


In [16]:
demographics_df.Gender.value_counts(dropna = False)

Gender
M      18814
F      15674
NaN        3
Name: count, dtype: int64

In [17]:
# Impute missing with most common sex (male)
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [18]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_AdvNSCLCBiomarkers.csv

In [19]:
biomarkers_df = processor.process_biomarkers(file_path = '../../aNSCLC/data/Enhanced_AdvNSCLCBiomarkers.csv',
                                             index_date_df = df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2026-05-06 00:17:03,454 - INFO - Successfully read Enhanced_AdvNSCLCBiomarkers.csv file with shape: (1021678, 18) and unique PatientIDs: 93715
2026-05-06 00:17:03,765 - INFO - Successfully merged Enhanced_AdvNSCLCBiomarkers.csv df with index_date_df resulting in shape: (245957, 19) and unique PatientIDs: 27290
2026-05-06 00:17:05,020 - INFO - Successfully processed Enhanced_AdvNSCLCBiomarkers.csv file with final shape: (34491, 11) and unique PatientIDs: 34491


In [20]:
biomarkers_df['PDL1_percent_staining'] = biomarkers_df["PDL1_percent_staining"].map({
    '0%': '0%', 
    '<1%': '1-49%',
    '1%': '1-49%',
    '2% - 4%': '1-49%',
    '5% - 9%': '1-49%', 
    '10% - 19%': '1-49%',
    '20% - 29%': '1-49%',
    '30% - 39%': '1-49%',
    '40% - 49%': '1-49%',
    '50% - 59%': '>=50%', 
    '60% - 69%': '>=50%', 
    '70% - 79%': '>=50%',
    '80% - 89%': '>=50%', 
    '90% - 99%': '>=50%', 
    '100%': '>=50%',
})

biomarkers_df['PDL1_percent_staining'] = biomarkers_df['PDL1_percent_staining'].fillna('unknown')
biomarkers_df['PDL1_percent_staining'] = biomarkers_df['PDL1_percent_staining'].astype('category')

In [21]:
biomarkers_df.PDL1_percent_staining.value_counts(dropna = False)

PDL1_percent_staining
unknown    31621
>=50%       1854
1-49%       1015
0%             1
Name: count, dtype: int64

In [22]:
biomarkers_df['EGFR_status'] = biomarkers_df['EGFR_status'].fillna('unknown')
biomarkers_df['ALK_status'] = biomarkers_df['ALK_status'].fillna('unknown')

In [23]:
biomarkers_df.EGFR_status.value_counts(dropna = False)

EGFR_status
negative    19343
unknown     14663
positive      485
Name: count, dtype: int64

In [24]:
biomarkers_df.ALK_status.value_counts(dropna = False)

ALK_status
negative    18664
unknown     15728
positive       99
Name: count, dtype: int64

### Process ECOG.csv

In [25]:
ecog_df = processor.process_ecog(file_path = '../../aNSCLC/data/ECOG.csv', 
                                 index_date_df = df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-05-06 00:17:05,599 - INFO - Successfully read ECOG.csv file with shape: (1623197, 4) and unique PatientIDs: 85577
2026-05-06 00:17:05,868 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (540160, 5) and unique PatientIDs: 26913
2026-05-06 00:17:06,318 - INFO - Successfully processed ECOG.csv file with final shape: (34491, 3) and unique PatientIDs: 34491


In [26]:
ecog_df.ecog_index.value_counts(dropna = False)

ecog_index
NaN    12167
1      10548
0       6324
2       4360
3       1037
4         55
Name: count, dtype: int64

In [27]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 1 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(1)

In [28]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [29]:
vitals_df = processor.process_vitals(file_path = '../../aNSCLC/data/Vitals.csv',
                                     index_date_df = df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2026-05-06 00:18:28,547 - INFO - Successfully read Vitals.csv file with shape: (29862758, 16) and unique PatientIDs: 114285
2026-05-06 00:18:40,874 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (9424178, 17) and unique PatientIDs: 34487
2026-05-06 00:18:45,898 - INFO - Successfully processed Vitals.csv file with final shape: (34491, 8) and unique PatientIDs: 34491


### Process Lab.csv

In [30]:
labs_df = processor.process_labs(file_path = '../../aNSCLC/data/Lab.csv',
                                 index_date_df = df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-05-06 00:21:43,213 - INFO - Successfully read Lab.csv file with shape: (82093498, 17) and unique PatientIDs: 108761
2026-05-06 00:22:27,833 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (27178870, 18) and unique PatientIDs: 34174
2026-05-06 00:23:31,258 - INFO - Successfully processed Lab.csv file with final shape: (34491, 76) and unique PatientIDs: 34491


### Process MedicationAdministration.csv

In [31]:
medications_df = processor.process_medications(file_path = '../../aNSCLC/data/MedicationAdministration.csv',
                                               index_date_df = df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2026-05-06 00:23:40,852 - INFO - Successfully read MedicationAdministration.csv file with shape: (7486734, 11) and unique PatientIDs: 89605
2026-05-06 00:23:42,967 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (2621877, 12) and unique PatientIDs: 33144
2026-05-06 00:23:43,164 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (34491, 9) and unique PatientIDs: 34491


### Process Diagnosis.csv

In [32]:
diagnosis_df = processor.process_diagnosis(file_path = '../../aNSCLC/data/Diagnosis.csv',
                                           index_date_df = df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2026-05-06 00:23:48,753 - INFO - Successfully read Diagnosis.csv file with shape: (9083016, 6) and unique PatientIDs: 114756
2026-05-06 00:23:49,956 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (2245409, 7) and unique PatientIDs: 34491
2026-05-06 00:23:55,454 - INFO - Successfully processed Diagnosis.csv file with final shape: (34491, 39) and unique PatientIDs: 34491


### Process Enhanced_Mortality_V2.csv

In [33]:
mortality_df = processor.process_mortality(file_path = '../../aNSCLC/data/Enhanced_Mortality_V2.csv',
                                           index_date_df = df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../../aNSCLC/data/Visit.csv', 
                                           telemedicine_path = '../../aNSCLC/data/Telemedicine.csv', 
                                           biomarkers_path = '../../aNSCLC/data/Enhanced_AdvNSCLCBiomarkers.csv', 
                                           oral_path = '../../aNSCLC/data/Enhanced_AdvNSCLC_Orals.csv',
                                           progression_path = '../../aNSCLC/data/Enhanced_AdvNSCLC_Progression.csv',
                                           drop_dates = False)

2026-05-06 00:23:55,547 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (84881, 2) and unique PatientIDs: 84881
2026-05-06 00:23:55,602 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (34491, 3) and unique PatientIDs: 34491
2026-05-06 00:23:59,826 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2026-05-06 00:23:59,847 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (34491, 6) and unique PatientIDs: 34491. There are 0 out of 34491 patients with missing duration values


In [34]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [35]:
mortality_df = mortality_df.query('duration >= 0')

### Process Insurance.csv

In [36]:
insurance_df = processor.process_insurance(file_path = '../../aNSCLC/data/Insurance.csv',
                                           index_date_df = df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0,
                                           missing_date_strategy = 'liberal')

/Users/xavierorcutt/Dropbox/io-heterogeneity/myenv/lib/python3.13/site-packages/flatiron_cleaner/general.py:1363: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
2026-05-06 00:24:00,468 - INFO - Successfully read Insurance.csv file with shape: (383622, 14) and unique PatientIDs: 106441
2026-05-06 00:24:00,682 - INFO - Successfully merged Insurance.csv df with index_date_df resulting in shape: (105724, 15) and unique PatientIDs: 31177
2026-05-06 00:24:00,989 - INFO - Successfully processed Insurance.csv file with final shape: (34491, 5) and unique PatientIDs: 34491


### Process SocialDeterminantsOfHealth.csv 

In [37]:
ses_df = pd.read_csv('../../aNSCLC/data/SocialDeterminantsOfHealth.csv')

In [38]:
ses_df.head(3)

,PatientID,SESIndex2015_2019
0,F9F11E927B585,5 - Highest SES
1,F998F69458EED,5 - Highest SES
2,F05B7A9AEA213,NaN


In [39]:
ses_df.SESIndex2015_2019.value_counts(dropna = False)

SESIndex2015_2019
4                  22426
3                  21632
2                  20561
5 - Highest SES    18784
1 - Lowest SES     18593
NaN                11618
Name: count, dtype: int64

In [40]:
ses_df['ses_mod'] = ses_df['SESIndex2015_2019'].map({
    '1 - Lowest SES' : 1, 
    '2': 2, 
    '3': 3, 
    '4': 4, 
    '5 - Highest SES': 5
})

ses_df['ses_mod_na'] = np.where(ses_df['ses_mod'].isna(), 1, 0)

# impute 3 for missing SES
ses_df['ses_mod'] = ses_df['ses_mod'].fillna(3)

In [41]:
ses_df = ses_df[['PatientID', 'ses_mod', 'ses_mod_na']]

In [42]:
ses_df = ses_df[ses_df.PatientID.isin(df.PatientID)]

## Merge dataframes

In [43]:
pembro_chemo_features_df = merge_dataframes(enhanced_df,
                                            demographics_df,
                                            biomarkers_df,
                                            ecog_df,
                                            vitals_df,
                                            labs_df,
                                            medications_df,
                                            diagnosis_df, 
                                            mortality_df, 
                                            insurance_df,
                                            ses_df,
                                            merge_type = 'inner')

2026-05-06 00:24:01,077 - INFO - Anticipated number of merges: 10
2026-05-06 00:24:01,077 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 164
2026-05-06 00:24:01,081 - INFO - Dataset 1 shape: (34491, 10), unique PatientIDs: 34491
2026-05-06 00:24:01,087 - INFO - Dataset 2 shape: (34491, 6), unique PatientIDs: 34491
2026-05-06 00:24:01,090 - INFO - Dataset 3 shape: (34491, 11), unique PatientIDs: 34491
2026-05-06 00:24:01,094 - INFO - Dataset 4 shape: (34491, 4), unique PatientIDs: 34491
2026-05-06 00:24:01,097 - INFO - Dataset 5 shape: (34491, 8), unique PatientIDs: 34491
2026-05-06 00:24:01,100 - INFO - Dataset 6 shape: (34491, 76), unique PatientIDs: 34491
2026-05-06 00:24:01,103 - INFO - Dataset 7 shape: (34491, 9), unique PatientIDs: 34491
2026-05-06 00:24:01,106 - INFO - Dataset 8 shape: (34491, 39), unique PatientIDs: 34491
2026-05-06 00:24:01,109 - INFO - Dataset 9 shape: (34299, 3), unique PatientIDs: 34299
2026-0

In [44]:
pembro_chemo_features_df.shape

(33828, 164)

In [45]:
pembro_chemo_features_df.head(2)

,PatientID,Histology,SmokingStatus,GroupStage_mod,days_diagnosis_to_adv,adv_diagnosis_year,days_adv_to_treatment,days_to_treatment_before_30d,GroupStage_mod_na,before_2020,...,other_viscera_met,other_met,event,duration,commercial,medicaid,medicare,other_insurance,ses_mod,ses_mod_na
0,F998F69458EED,Non-squamous cell carcinoma,History of smoking,4.0,0.0,2023,23,1,0,0,...,0,0,0,679.0,1,0,1,0,5.0,0
1,F05B7A9AEA213,Non-squamous cell carcinoma,History of smoking,3.0,388.0,2019,12,1,0,1,...,0,0,1,259.0,1,0,0,1,3.0,1


In [46]:
pembro_chemo_features_df = pembro_chemo_features_df.query('PDL1_percent_staining == ">=50%"')
pembro_chemo_features_df = pembro_chemo_features_df.query('EGFR_status == "negative" or EGFR_status == "unknown"')
pembro_chemo_features_df = pembro_chemo_features_df.query('ALK_status == "negative" or ALK_status == "unknown"')

In [47]:
pembro_chemo_features_df.shape

(1803, 164)

## Export dataframe

In [48]:
pembro_chemo_features_df.to_csv('../outputs/pembro_chemo_features_df.csv', index = False)

In [49]:
# Save dtypes
pembro_chemo_features_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/pembro_chemo_features_dtypes.csv')